### Experiment - dataset annotation sampling

In [1]:
from utils import load_pkl
from pathlib import Path
import pandas as pd

In [2]:
pwd = Path.cwd()
data_dir = pwd.parent / 'data'
output_dir = pwd / 'samples'
filepath = data_dir / 'easdrl' / 'refined_cooking_data.pkl'

indata = load_pkl(filepath)[-1]

print(indata[0])
sents = [l['this_sent'] for l in indata[0]]

[{'this_sent': ['MAKE', 'breakfast', 'the', 'night', 'before', 'with', 'a', 'hash', 'brown', 'casserole'], 'acts': [{'act_idx': 0, 'obj_idxs': [[1], []], 'act_type': 1, 'related_acts': []}], 'last_sent': []}, {'this_sent': ['Start', 'by', 'mixing', 'together', 'your', 'cream', 'of', 'mushroom', 'condensed', 'soup', 'sour', 'cream', 'salt', 'pepper', 'onion', 'and', 'cheese', 'in', 'a', 'large', 'bowl'], 'acts': [{'act_idx': 12, 'obj_idxs': [[15, 19, 21, 22, 23, 24, 26], []], 'act_type': 1, 'related_acts': []}], 'last_sent': ['Make', 'breakfast', 'the', 'night', 'before', 'with', 'a', 'hash', 'brown', 'casserole']}, {'this_sent': ['After', 'these', 'are', 'well', 'combined', 'add', 'in', 'the', 'hash', 'browns', 'and', 'mix', 'gently'], 'acts': [{'act_idx': 25, 'obj_idxs': [[5, 9, 11, 12, 13, 14, 16], []], 'act_type': 1, 'related_acts': []}, {'act_idx': 26, 'obj_idxs': [[30], []], 'act_type': 1, 'related_acts': []}, {'act_idx': 32, 'obj_idxs': [[5, 9, 11, 12, 13, 14, 16, 30], []], 'act_

### Parameters
parameters in sampling

In [3]:
n_docs_per_dataset = 10 # Number of documents to sample per dataset
rand_seed = 42

In [4]:
def get_obj_word(sent, obj_idxs):
    words = []
    for idx in obj_idxs:
        if idx != -1:
            words.append(sent[idx])
        else:
            words.append("UNKNOWN")
    return words

def get_action_annotations(row):
    this_sent = row["this_sent_tokens"]
    last_sent = row["last_sent_tokens"]

    whole_sent = last_sent + this_sent if last_sent else this_sent

    act_idx = row["act_idx"]
    act_name = whole_sent[act_idx]

    obj_idxs = row["obj_idxs"]
    if len(obj_idxs) != 2:
        raise ValueError(f"Expected obj_idxs to have length 2, but got {len(obj_idxs)}")

    main_obj_idxs = row["obj_idxs"][0]
    opt_obj_idxs = row["obj_idxs"][1]

    main_obj_names = get_obj_word(whole_sent, main_obj_idxs)
    
    if opt_obj_idxs:
        opt_obj_names = get_obj_word(whole_sent, opt_obj_idxs)
        return f"{act_name} ({','.join(main_obj_names)}); {act_name} ({','.join(opt_obj_names)})"
    
    return f"{act_name} ({','.join(main_obj_names)})"
    

def get_df_from_pkl(indata, dataset_name):
    df = pd.DataFrame([
        {
            "dataset": dataset_name,
            "doc_id": doc_id,
            "sent_id": sent_id,
            "this_sent": " ".join(step["this_sent"]),
            "last_sent": " ".join(step["last_sent"]),
            "this_sent_tokens": step["this_sent"],
            "last_sent_tokens": step["last_sent"],
            "act_idx": act["act_idx"],
            "obj_idxs": act["obj_idxs"],
            "act_type": act["act_type"],
            "related_acts": act["related_acts"],
        }
        for doc_id, doc in enumerate(indata)
        for sent_id, step in enumerate(doc)
        if step and "acts" in step
        for act in step["acts"]
    ])

    df["action_annotations"] = df.apply(get_action_annotations, axis=1)
    return df

Create dataframe based on pkl file, each row contains one action

In [5]:
cooking_dataset_path = data_dir / 'easdrl' / 'refined_cooking_data.pkl'
wikihow_dataset_path = data_dir / 'easdrl' / 'refined_wikihow_data.pkl'
win2k_dataset_path = data_dir / 'easdrl' / 'refined_win2k_data.pkl'

cooking_indata = load_pkl(cooking_dataset_path)[-1]
wikihow_indata = load_pkl(wikihow_dataset_path)[-1]
win2k_indata = load_pkl(win2k_dataset_path)[-1]

cooking_df = get_df_from_pkl(cooking_indata, 'cooking')
wikihow_df = get_df_from_pkl(wikihow_indata, 'wikihow')
win2k_df = get_df_from_pkl(win2k_indata, 'win2k')

combined_df = pd.concat([cooking_df, wikihow_df, win2k_df], ignore_index=True)
combined_df.head()

,dataset,doc_id,sent_id,this_sent,last_sent,this_sent_tokens,last_sent_tokens,act_idx,obj_idxs,act_type,related_acts,action_annotations
0,cooking,0,0,MAKE breakfast the night before with a hash br...,,"[MAKE, breakfast, the, night, before, with, a,...",[],0,"[[1], []]",1,[],MAKE (breakfast)
1,cooking,0,1,Start by mixing together your cream of mushroo...,Make breakfast the night before with a hash br...,"[Start, by, mixing, together, your, cream, of,...","[Make, breakfast, the, night, before, with, a,...",12,"[[15, 19, 21, 22, 23, 24, 26], []]",1,[],"mixing (cream,soup,cream,salt,pepper,onion,che..."
2,cooking,0,2,After these are well combined add in the hash ...,Start by mixing together your cream of mushroo...,"[After, these, are, well, combined, add, in, t...","[Start, by, mixing, together, your, cream, of,...",25,"[[5, 9, 11, 12, 13, 14, 16], []]",1,[],"combined (cream,soup,cream,salt,pepper,onion,c..."
3,cooking,0,2,After these are well combined add in the hash ...,Start by mixing together your cream of mushroo...,"[After, these, are, well, combined, add, in, t...","[Start, by, mixing, together, your, cream, of,...",26,"[[30], []]",1,[],add (browns)
4,cooking,0,2,After these are well combined add in the hash ...,Start by mixing together your cream of mushroo...,"[After, these, are, well, combined, add, in, t...","[Start, by, mixing, together, your, cream, of,...",32,"[[5, 9, 11, 12, 13, 14, 16, 30], []]",1,[],"mix (cream,soup,cream,salt,pepper,onion,cheese..."


Sample n documents per dataset

In [6]:
sampled_doc_ids = (
    combined_df[["dataset", "doc_id"]]
    .drop_duplicates()
    .groupby("dataset", group_keys=False)
    .sample(n=n_docs_per_dataset, random_state=rand_seed)
    .sort_values(["dataset", "doc_id"])
    .reset_index(drop=True)
)

sampled_annotation_df = (
    combined_df
    .merge(sampled_doc_ids, on=["dataset", "doc_id"], how="inner")
    .sort_values(["dataset", "doc_id", "sent_id", "act_idx"])
    .reset_index(drop=True)
)

display(sampled_doc_ids.groupby("dataset").size())
sampled_annotation_df.head()

dataset
cooking    10
wikihow    10
win2k      10
dtype: int64

,dataset,doc_id,sent_id,this_sent,last_sent,this_sent_tokens,last_sent_tokens,act_idx,obj_idxs,act_type,related_acts,action_annotations
0,cooking,4,0,How to make an English flapjack,,"[How, to, make, an, English, flapjack]",[],2,"[[5], []]",1,[],make (flapjack)
1,cooking,4,1,Preheat your oven at 350 degree C and prepare ...,How to make an English flapjack,"[Preheat, your, oven, at, 350, degree, C, and,...","[How, to, make, an, English, flapjack]",6,"[[8], []]",1,[],Preheat (oven)
2,cooking,4,1,Preheat your oven at 350 degree C and prepare ...,How to make an English flapjack,"[Preheat, your, oven, at, 350, degree, C, and,...","[How, to, make, an, English, flapjack]",14,"[[19], []]",1,[],prepare (pan)
3,cooking,4,2,You may also use non-stick baking paper which ...,Preheat your oven at 350 degree C and prepare ...,"[You, may, also, use, non-stick, baking, paper...","[Preheat, your, oven, at, 350, degree, C, and,...",17,"[[20], []]",1,[],use (paper)
4,cooking,4,3,In a bowl mix the butter 4 tablespoons of syru...,You may also use non-stick baking paper which ...,"[In, a, bowl, mix, the, butter, 4, tablespoons...","[You, may, also, use, non-stick, baking, paper...",18,"[[20, 24, 26], []]",1,[],"mix (butter,syrup,sugar)"


divide the sampled documents into 2 groups for manual inspectation

In [7]:
n_docs_per_group = n_docs_per_dataset // 2

sampled_doc_ids_with_group = (
    sampled_doc_ids
    .groupby("dataset", group_keys=False)
    .sample(frac=1, random_state=rand_seed)
    .reset_index(drop=True)
)
sampled_doc_ids_with_group["group"] = (
    sampled_doc_ids_with_group
    .groupby("dataset")
    .cumcount()
    .map(lambda i: "A" if i < n_docs_per_group else "B")
)
sampled_doc_ids_with_group = sampled_doc_ids_with_group.sort_values(["dataset", "group", "doc_id"])

sampled_annotation_df = sampled_annotation_df.merge(
    sampled_doc_ids_with_group,
    on=["dataset", "doc_id"],
    how="left",
)

sampled_A = sampled_annotation_df[sampled_annotation_df["group"] == "A"].reset_index(drop=True)
sampled_B = sampled_annotation_df[sampled_annotation_df["group"] == "B"].reset_index(drop=True)

def write_sample_preserving_existing_columns(sample_df, output_path):
    sample_df = sample_df.copy()

    if output_path.exists():
        existing_df = pd.read_csv(output_path)
        if len(existing_df) != len(sample_df):
            raise ValueError(
                f"{output_path.name} has {len(existing_df)} rows, "
                f"but the current sample has {len(sample_df)} rows."
            )

        existing_extra_cols = [
            col for col in existing_df.columns
            if col not in sample_df.columns
        ]
        output_df = sample_df.copy()
        for col in existing_extra_cols:
            output_df[col] = existing_df[col].reset_index(drop=True)

        if existing_extra_cols:
            print(f"Preserved existing columns from {output_path.name}: {', '.join(existing_extra_cols)}")
    else:
        output_df = sample_df
        output_df["label_correctness"] = 0

    output_df.to_csv(output_path, index=False)


output_dir.mkdir(parents=True, exist_ok=True)
write_sample_preserving_existing_columns(sampled_A, output_dir / "sampled_A.csv")
write_sample_preserving_existing_columns(sampled_B, output_dir / "sampled_B.csv")

display(sampled_doc_ids_with_group.groupby(["dataset", "group"]).size())

Preserved existing columns from sampled_A.csv: label_correctness, error_type
Preserved existing columns from sampled_B.csv: label_correctness, error_type


dataset  group
cooking  A        5
         B        5
wikihow  A        5
         B        5
win2k    A        5
         B        5
dtype: int64